# Final Pipeline Notebook

This notebook implements the full project lifecycle for diabetic 30-day readmission risk analysis. It imports reusable functions from `src/processing.py`, performs web scraping for medication augmentation, audits the combined dataset, conducts exploratory analysis, and trains a modeling pipeline.

In [10]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd()
if not (project_root / 'src').exists() and (project_root.parent / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.processing import (
    MEDICATION_COLUMNS,
    augment_with_medication_classes,
    analyze_change_effect,
    audit_data,
    build_medication_class_table,
    clean_diabetes_data,
    load_data,
    save_cleaned_data,
    scrape_medication_summaries_parallel,
    scrape_medication_summaries_sequential,
    train_readmission_model,
)

print(f'Using project root: {project_root}')

plt.rcParams['figure.figsize'] = (10, 6)
plt.style.use('seaborn-v0_8')

Using project root: c:\Users\laila\OneDrive\Desktop\health-project


## Load Dataset from Data Folder

Read the dataset from `data/diabetic_data.csv` and confirm it loads correctly.

In [13]:
raw_df = load_data('../data/diabetic_data.csv')
print(f'Loaded dataset with {raw_df.shape[0]} rows and {raw_df.shape[1]} columns.')
raw_df.head(5)

Loaded dataset with 101766 rows and 50 columns.


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Inspect Dataset Structure

Display the first rows, the data shape, data types, and summary statistics for numeric columns.

In [ ]:
print('Shape:', raw_df.shape)
print('\nData types:')
print(raw_df.dtypes.head(20))

raw_df.describe(include='all').loc[['count', 'unique', 'top', 'freq']].T.head(10)

## Clean and Prepare Data

Transform columns into model-ready formats, encode medication flags, and prepare the target variable for 30-day readmission prediction.

In [ ]:
clean_df = clean_diabetes_data(raw_df)
print('Cleaned shape:', clean_df.shape)
clean_df[['age_years', 'change_flag', 'diabetesMed_flag', 'readmitted_30']].head(5)

## Data Scraping & Augmentation

Use web scraping to collect supplementary medication information and compare sequential vs. parallel scraping performance.

In [ ]:
print('Medication columns to augment:')
print(MEDICATION_COLUMNS)

sequential_summaries, sequential_time = scrape_medication_summaries_sequential(MEDICATION_COLUMNS)
parallel_summaries, parallel_time = scrape_medication_summaries_parallel(MEDICATION_COLUMNS)

print(f'Sequential scraping time: {sequential_time:.2f} seconds')
print(f'Parallel scraping time:   {parallel_time:.2f} seconds')

med_class_table = build_medication_class_table(parallel_summaries)
med_class_table.head(10)

In [ ]:
augmented_df = augment_with_medication_classes(clean_df, med_class_table)
print('Augmented shape:', augmented_df.shape)
augmented_df[[c for c in augmented_df.columns if c.endswith('_count')]].head(5)

## Data Audit

Assess the augmented dataset for missing values, outliers, and schema issues before modeling.

In [ ]:
audit_summary = audit_data(augmented_df)
print('Dataset shape:', audit_summary['shape'])
print('\nMissing values (top 20):')
print(pd.Series(audit_summary['missing_counts']).sort_values(ascending=False).head(20))
print('\nOutlier counts (numeric):')
print(pd.Series(audit_summary['outlier_counts']).sort_values(ascending=False).head(20))

## Exploratory Data Analysis

Compute correlations and use grouping to compare readmission outcomes and the effect of medication changes.

In [ ]:
corr_cols = ['age_years', 'time_in_hospital', 'num_lab_procedures', 'num_medications', 'number_inpatient', 'number_diagnoses', 'change_flag', 'readmitted_30']
print(augmented_df[corr_cols].corr()['readmitted_30'].sort_values(ascending=False))

readmission_by_change = augmented_df.groupby('change_flag')['readmitted_30'].mean().rename({0: 'No Change', 1: 'Change'})
readmission_by_change

## Visualize Key Features

Plot distributions and the relationship between medication change and readmission risk.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
augmented_df['readmitted_30'].value_counts(normalize=True).plot(kind='bar', ax=ax[0])
ax[0].set_title('Readmission <30 days Rate')
ax[0].set_ylabel('Proportion')

readmission_by_change.plot(kind='bar', ax=ax[1], color=['#4c72b0', '#dd8452'])
ax[1].set_title('Readmission Rate by Medication Change Flag')
ax[1].set_ylabel('Proportion')
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(augmented_df.loc[augmented_df['readmitted_30'] == 1, 'age_years'].dropna(), bins=15, alpha=0.7, label='<30 readmit')
plt.hist(augmented_df.loc[augmented_df['readmitted_30'] == 0, 'age_years'].dropna(), bins=15, alpha=0.7, label='No <30 readmit')
plt.title('Age Distribution by Readmission Group')
plt.xlabel('Age (years)')
plt.ylabel('Count')
plt.legend()
plt.show()

## Modeling Pipeline

Train a readmission risk model using `sklearn.pipeline`, cross-validation, and hyperparameter tuning.

In [ ]:
best_model, model_metrics = train_readmission_model(augmented_df)
print('Best model parameters:')
print(model_metrics['best_params'])
print('\nROC AUC:', model_metrics['roc_auc'])
print('\nConfusion matrix:')
print(model_metrics['confusion_matrix'])
print('\nClassification report:')
print(pd.DataFrame(model_metrics['classification_report']).T)

## Medication Change Effect Analysis

Analyze whether medication changes during admission correlate with elevated 30-day readmission risk after controlling for patient severity.

In [ ]:
change_effect = analyze_change_effect(augmented_df)
print('Change flag odds ratio:', change_effect['change_flag_odds_ratio'])
print('Logistic coefficients:')
print(change_effect['coefficients'])

## Save Processed Data

Export the cleaned and augmented dataset for future use and reproducibility.

In [ ]:
cleaned_path = save_cleaned_data(augmented_df, 'data/diabetic_data_cleaned.csv')
print(f'Cleaned dataset saved to: {cleaned_path}')